# KrishiSetu AI - Odisha Crop Disease Classifier
## Vertex AI Training + Google AI Integration

---

### Hackathon Compliance
This notebook integrates Google AI tools as required:
- **Vertex AI** - Model training and deployment
- **Gemini API** - Cloud-based crop diagnosis
- **Google Cloud TTS** - Voice readout in local languages
- **Google Translation API** - Multilingual support

### Focus: Odisha Crops
- Paddy (Rice) - Main crop
- Maize (Corn) - Sundargarh, Rayagada
- Cotton - Kalahandi
- Tomato, Potato - Widely grown

### VERIFIED Datasets

| # | Dataset | URL | Size |
|---|---------|-----|------|
| 1 | PlantVillage | [kaggle.com/datasets/emmarex/plantdisease](https://www.kaggle.com/datasets/emmarex/plantdisease) | 54K images |
| 2 | Rice Disease | [kaggle.com/datasets/anshulm257/rice-disease-dataset](https://www.kaggle.com/datasets/anshulm257/rice-disease-dataset) | 3,829 images |
| 3 | Cotton Leaf Disease | [kaggle.com/datasets/seroshkarim/cotton-leaf-disease-dataset](https://www.kaggle.com/datasets/seroshkarim/cotton-leaf-disease-dataset) | 1,710 images |

### Step-by-Step Guide

#### Step 1: Open Google Colab
1. Go to [colab.research.google.com](https://colab.research.google.com)
2. Click File > Upload notebook
3. Upload this file

#### Step 2: Enable GPU
1. Runtime > Change runtime type
2. Select T4 GPU
3. Save

#### Step 3: Connect to Google Cloud (Vertex AI)
1. Run Cell 1 to install packages
2. Run Cell 2 to authenticate with Google Cloud
3. You'll be asked to log in with your Google account
4. Copy the auth code and paste it

#### Step 4: Download Datasets

**A. Rice Disease Dataset:**
1. Go to: https://www.kaggle.com/datasets/anshulm257/rice-disease-dataset
2. Click Download
3. Upload zip to Colab Files sidebar

**B. Cotton Leaf Disease:**
1. Go to: https://www.kaggle.com/datasets/seroshkarim/cotton-leaf-disease-dataset
2. Click Download
3. Upload zip to Colab Files sidebar

#### Step 5: Run All Cells
Runtime > Run all

#### Step 6: Deploy to Vertex AI
After training, the model is automatically uploaded to Vertex AI Model Registry.

#### Step 7: Download TFJS Model
1. Run Section 11
2. Download tfjs_model.zip
3. Extract to KrishiSetu-AI/public/model/
---

> **Colab environment note:** tested on a fresh T4 runtime with Python 3.13 / numpy 2.x. Do NOT downgrade numpy - cell 1 adds the compatibility shim tensorflowjs needs instead.
> **Preprocessing contract:** the web app sends pixels as [0,1] (divides by 255). The data pipeline normalizes to [0,1] and a `Rescaling(2, -1)` layer is baked into the model. Do not remove either, or offline scanning breaks.


---
## 1. Install Packages

In [ ]:
# ============================================================
# 1. Install packages (Colab-compatible - NO numpy downgrade)
# ============================================================
# Colab (Python 3.13) ships numpy 2.x plus a TensorFlow build that matches it.
# Downgrading numpy here breaks scipy/tensorflow (the '_blas_supports_fpe'
# error you may have seen), so we do NOT touch numpy.
# tensorflowjs still references the removed np.object/np.bool aliases, so we
# re-add them below BEFORE importing it.

!pip install -q -U tensorflowjs opencv-python-headless matplotlib seaborn scikit-learn requests
!pip install -q google-cloud-aiplatform google-cloud-storage

# --- numpy alias shim (MUST run before importing tensorflowjs) ---
import numpy as np
if not hasattr(np, 'object'):
    np.object = object
if not hasattr(np, 'bool'):
    np.bool = bool
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'str'):
    np.str = str

import os, sys, json, time, shutil, zipfile, re, requests, random
import tensorflow as tf
import tensorflowjs as tfjs
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print(f'NumPy:         {np.__version__}')
print(f'TensorFlow:    {tf.__version__}')
print(f'TensorFlow.js: {tfjs.__version__}')
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('If imports succeeded here, the environment is ready.')


---
## 1b. Create project folders

Creates every working folder this notebook needs under `/content`.
Re-running this cell is always safe. In YOUR repo you will later create
`public/model` (command shown again at the end).


In [ ]:
# ============================================================
# 1b. Create all working folders (safe to re-run)
# ============================================================
FOLDERS = [
    '/content/rice_dataset',    # extracted rice Kaggle zip
    '/content/cotton_dataset',  # extracted cotton Kaggle zip
    '/content/odisha_crops',    # final training dataset (Crop_Disease folders)
    '/content/tfjs_model',      # exported TF.js model (model.json + .bin)
]
for d in FOLDERS:
    os.makedirs(d, exist_ok=True)
    print(f'  ready: {d}')

# NOTE: /content/plantvillage is NOT pre-created - a later cell auto-downloads there.
# NOTE: in YOUR repo (not Colab) you will later run:  mkdir -p public/model

print()
print('All working folders ready.')


---
## 2. Authenticate with Google Cloud (Vertex AI)

**IMPORTANT:** You need a Google Cloud project with Vertex AI enabled.

### Setup Instructions:
1. Go to [console.cloud.google.com](https://console.cloud.google.com)
2. Create a new project or select existing
3. Enable Vertex AI API:
   - Go to APIs & Services > Library
   - Search for "Vertex AI API"
   - Click Enable
4. Create a service account:
   - Go to IAM & Admin > Service Accounts
   - Create Service Account
   - Role: Vertex AI Admin
   - Create key (JSON)
   - Download the JSON key file
5. Upload the JSON key to Colab Files sidebar
6. Set PROJECT_ID below

In [ ]:
# ============================================================
# Authenticate with Google Cloud
# ============================================================

# OPTION 1: Use service account key (recommended)
# Upload your service account JSON to Colab, then set path here
SERVICE_ACCOUNT_KEY = '/content/service-account.json'  # CHANGE THIS

import os
import json

if os.path.exists(SERVICE_ACCOUNT_KEY):
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = SERVICE_ACCOUNT_KEY
    with open(SERVICE_ACCOUNT_KEY) as f:
        key_data = json.load(f)
    PROJECT_ID = key_data.get('project_id', 'your-project-id')
    print(f'Authenticated with project: {PROJECT_ID}')
else:
    print('WARNING: Service account key not found!')
    print('Upload your service account JSON to Colab and update SERVICE_ACCOUNT_KEY path')
    PROJECT_ID = 'your-project-id'  # CHANGE THIS

# OPTION 2: Use Colab built-in auth (simpler but less secure)
# from google.colab import auth
# auth.authenticate_user()
# PROJECT_ID = 'your-project-id'  # CHANGE THIS

print(f'Project ID: {PROJECT_ID}')
print('If you see errors, make sure Vertex AI API is enabled in your project.')

---
## 3. Initialize Vertex AI

In [ ]:
# ============================================================
# Initialize Vertex AI
# ============================================================

from google.cloud import aiplatform

aiplatform.init(
    project=PROJECT_ID,
    location='us-central1',
    staging_bucket=f'gs://{PROJECT_ID}-krishisetu'
)

print(f'Vertex AI initialized')
print(f'Project: {PROJECT_ID}')
print(f'Region: us-central1')
print(f'Staging bucket: gs://{PROJECT_ID}-krishisetu')

---
## 4. Download PlantVillage Dataset (Auto)

In [ ]:
# ============================================================
# Auto-download PlantVillage dataset
# ============================================================

pv_dir = '/content/plantvillage'

if not os.path.exists(pv_dir):
    print('Downloading PlantVillage dataset (~150MB)...')
    url = 'https://storage.googleapis.com/plantvillage-dataset/PlantVillage.zip'
    zip_path = '/content/plantvillage.zip'
    
    try:
        r = requests.get(url, stream=True, timeout=300)
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        downloaded = 0
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded += len(chunk)
                if total > 0:
                    pct = (downloaded / total) * 100
                    print(f'\r  Downloading: {pct:.1f}%', end='')
        print(f'\n  Downloaded: {os.path.getsize(zip_path) / 1e6:.1f} MB')
        
        print('  Extracting...')
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('/content/')
        os.remove(zip_path)
        print('  Done!')
    except Exception as e:
        print(f'  Auto-download failed: {e}')
        print('\n  MANUAL: Download from https://www.kaggle.com/datasets/emmarex/plantdisease')
else:
    print(f'PlantVillage already exists at {pv_dir}')

# Show what we got
if os.path.exists(pv_dir):
    classes = sorted([d for d in os.listdir(pv_dir) if os.path.isdir(os.path.join(pv_dir, d))])
    print(f'\nFound {len(classes)} classes in PlantVillage:')
    for c in classes:
        count = len(os.listdir(os.path.join(pv_dir, c)))
        print(f'  {c}: {count} images')

---
## 5. Upload Kaggle Datasets

In [ ]:
# ============================================================
# Extract uploaded Kaggle datasets
# ============================================================

# Check for uploaded rice dataset
rice_files = [f for f in os.listdir('/content') if 'rice' in f.lower() and f.endswith('.zip')]
cotton_files = [f for f in os.listdir('/content') if 'cotton' in f.lower() and f.endswith('.zip')]

if rice_files:
    print(f'Found rice dataset: {rice_files[0]}')
    with zipfile.ZipFile(f'/content/{rice_files[0]}', 'r') as z:
        z.extractall('/content/rice_dataset')
    print('  Extracted to /content/rice_dataset/')
else:
    print('WARNING: No rice dataset found!')
    print('  Download: https://www.kaggle.com/datasets/anshulm257/rice-disease-dataset')

if cotton_files:
    print(f'Found cotton dataset: {cotton_files[0]}')
    with zipfile.ZipFile(f'/content/{cotton_files[0]}', 'r') as z:
        z.extractall('/content/cotton_dataset')
    print('  Extracted to /content/cotton_dataset/')
else:
    print('WARNING: No cotton dataset found!')
    print('  Download: https://www.kaggle.com/datasets/seroshkarim/cotton-leaf-disease-dataset')

---
## 6. Build Odisha Crop Dataset

In [ ]:
# ============================================================
# 6. Build the Odisha dataset (keyword-based class mapping)
# ============================================================
# Creates /content/odisha_crops/<Crop_Disease>/ folders and copies images
# into them. Folder names use the required Crop_Disease format because
# classes.json is generated from them later.

odisha_dir = '/content/odisha_crops'
os.makedirs(odisha_dir, exist_ok=True)

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
CAP_PER_CLASS = 1500  # cap per class to keep training balanced/fast


def leaf_image_folders(root, max_depth=4):
    """Directories that contain images but no sub-directory with images."""
    out = []
    if not os.path.isdir(root):
        return out
    base_depth = os.path.abspath(root).rstrip('/').count('/')
    for dirpath, dirnames, filenames in os.walk(root):
        depth = os.path.abspath(dirpath).rstrip('/').count('/') - base_depth
        if depth > max_depth:
            dirnames[:] = []
            continue
        has_img = any(f.lower().endswith(IMG_EXTS) for f in filenames)
        if has_img:
            # leaf = no direct child directory also contains images
            children_with_images = False
            for d in dirnames:
                child = os.path.join(dirpath, d)
                if any(f.lower().endswith(IMG_EXTS) for f in os.listdir(child)) if os.path.isdir(child) else False:
                    children_with_images = True
                    break
            if not children_with_images:
                out.append(dirpath)
    return out


def classify(path):
    """Map a source folder path to an Odisha class name (Crop_Disease) or None."""
    name = os.path.basename(os.path.normpath(path))
    parent = os.path.basename(os.path.dirname(os.path.normpath(path)))
    text = re.sub(r'\s+', ' ', (name + ' ' + parent).lower()).replace('_', ' ')
    text = text.replace('(maize)', ' maize ')

    # 1) crop
    if re.search(r'rice|paddy', text):
        crop = 'Paddy'
    elif re.search(r'corn|maize', text):
        crop = 'Maize'
    elif re.search(r'tomato', text):
        crop = 'Tomato'
    elif re.search(r'potato', text):
        crop = 'Potato'
    elif re.search(r'cotton', text):
        crop = 'Cotton'
    else:
        return None

    # 2) disease
    dis = None
    if re.search(r'bacterial.*blight', text):
        dis = 'Bacterial_Blight'
    elif re.search(r'brown.*spot', text):
        dis = 'Brown_Spot'
    elif re.search(r'blast', text):
        dis = 'Blast'
    elif re.search(r'leaf.?smut', text):
        dis = 'Leaf_Smut'
    elif re.search(r'tungro', text):
        dis = 'Tungro'
    elif re.search(r'hispa', text):
        dis = 'Hispa'
    elif re.search(r'sheath', text):
        dis = 'Sheath_Blight'
    elif re.search(r'northern.*blight', text):
        dis = 'Leaf_Blight'
    elif re.search(r'gray.*leaf|cercospora', text):
        dis = 'Gray_Leaf_Spot'
    elif re.search(r'common.*rust|(^|[^a-z])rust', text):
        dis = 'Rust'
    elif re.search(r'bacterial.*spot', text):
        dis = 'Bacterial_Spot'
    elif re.search(r'early.*blight', text):
        dis = 'Early_Blight'
    elif re.search(r'late.*blight', text):
        dis = 'Late_Blight'
    elif re.search(r'leaf.?mold', text):
        dis = 'Leaf_Mold'
    elif re.search(r'septoria', text):
        dis = 'Septoria'
    elif re.search(r'spider.?mite', text):
        dis = 'Spider_Mite'
    elif re.search(r'target.*spot', text):
        dis = 'Target_Spot'
    elif re.search(r'yellow.*curl|curl.*virus', text):
        dis = 'Yellow_Leaf_Curl'
    elif re.search(r'leaf.?curl', text):
        dis = 'Leaf_Curl'
    elif re.search(r'mosaic', text):
        dis = 'Mosaic'
    elif re.search(r'fusarium|wilt', text):
        dis = 'Fusarium_Wilt'
    elif re.search(r'healthy|fresh', text):
        dis = 'Healthy'

    if crop and dis:
        return f'{crop}_{dis}'
    return None


# collect candidate class folders from every dataset root
roots = [
    '/content/plantvillage',
    '/content/PlantVillage',
    '/content/plantvillage_raw',
    '/content/rice_dataset',
    '/content/cotton_dataset',
]
sources = []
for r in roots:
    sources.extend(leaf_image_folders(r))

mapped = {}
for src in sources:
    target = classify(src)
    if target:
        mapped.setdefault(target, []).append(src)

print(f'Matched {len(mapped)} classes from {len(sources)} source folders\n')

total = 0
for target in sorted(mapped):
    dst = os.path.join(odisha_dir, target)
    os.makedirs(dst, exist_ok=True)   # <-- folder creation per class
    count = 0
    for src_dir in mapped[target]:
        for img in sorted(os.listdir(src_dir)):
            if count >= CAP_PER_CLASS:
                break
            if img.lower().endswith(IMG_EXTS):
                shutil.copy2(os.path.join(src_dir, img),
                             os.path.join(dst, f'{target}_{count:05d}.jpg'))
                count += 1
        if count >= CAP_PER_CLASS:
            break
    total += count
    print(f'  {target:28s} {count:5d} images  ->  {dst}')

# prune any class folder that ended up empty
for d in sorted(os.listdir(odisha_dir)):
    p = os.path.join(odisha_dir, d)
    if os.path.isdir(p) and not os.listdir(p):
        os.rmdir(p)
        print(f'  removed empty class folder: {d}')

print(f'\nTotal: {total} images across {len(os.listdir(odisha_dir))} classes')
print(f'Dataset root (created): {odisha_dir}')


---
## 7. Data Loading & Augmentation

In [ ]:
# ============================================================
# 7. Data loading + augmentation (normalized to [0,1] like the app)
# ============================================================
# The web app divides pixels by 255 before inference ([0,1] range), so the
# training data must be [0,1] too. A Rescaling(2, -1) layer inside the model
# (next section) then maps [0,1] -> [-1,1] for MobileNetV2.

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
VAL_SPLIT = 0.2

train_ds = tf.keras.utils.image_dataset_from_directory(
    odisha_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset='training'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    odisha_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    seed=SEED,
    validation_split=VAL_SPLIT,
    subset='validation'
)

class_names = train_ds.class_names
num_classes = len(class_names)


def normalize(x, y):
    return x / 255.0, y


data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomBrightness(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name='data_augmentation')


def augment(x, y):
    return data_augmentation(x, training=True), y


train_ds = (train_ds
            .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
            .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))
val_ds = val_ds.map(normalize, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

print(f'\nClasses ({num_classes}): {class_names}')
for xb, yb in train_ds.take(1):
    print('Sample pixel range after normalize (expect ~0.0 - 1.0):',
          float(xb.numpy().min()), '-', float(xb.numpy().max()))


---
## 8. Build Model (MobileNetV2)

In [ ]:
# ============================================================
# 8. Build model (MobileNetV2 transfer learning)
# ============================================================
# The Rescaling(2, -1) layer converts the app's [0,1] input to [-1,1]
# (the range MobileNetV2 imagenet weights expect). It is baked into the
# exported model.json - the app must NOT be changed. DO NOT REMOVE IT.

from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
    alpha=1.0
)
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(scale=2.0, offset=-1.0, name='input_rescale'),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation='softmax')
], name='krishisetu_odisha')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
total_params = model.count_params()
print(f'\nTotal parameters: {total_params:,}')


---
## 9. Train the Model

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=2, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1
    )
]

print('=' * 60)
print('PHASE 1: Training classifier head (frozen base)')
print('=' * 60)

history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=10, callbacks=callbacks, verbose=1
)

print(f'\nPhase 1 best val accuracy: {max(history1.history["val_accuracy"]):.4f}')

In [ ]:
print('=' * 60)
print('PHASE 2: Fine-tuning top MobileNetV2 layers')
print('=' * 60)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=5, callbacks=callbacks, verbose=1
)

print(f'\nPhase 2 best val accuracy: {max(history2.history["val_accuracy"]):.4f}')

---
## 10. Evaluate

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

all_acc = history1.history['accuracy'] + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss = history1.history['loss'] + history2.history['loss']
all_val_loss = history1.history['val_loss'] + history2.history['val_loss']

epochs = range(1, len(all_acc) + 1)

ax1.plot(epochs, all_acc, 'b-o', label='Train', markersize=4)
ax1.plot(epochs, all_val_acc, 'r-o', label='Val', markersize=4)
ax1.axvline(x=len(history1.history['accuracy']), color='gray', linestyle='--', alpha=0.5)
ax1.set_title('Accuracy', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, all_loss, 'b-o', label='Train', markersize=4)
ax2.plot(epochs, all_val_loss, 'r-o', label='Val', markersize=4)
ax2.axvline(x=len(history1.history['loss']), color='gray', linestyle='--', alpha=0.5)
ax2.set_title('Loss', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print('\n' + '=' * 60)
print('CLASSIFICATION REPORT - Odisha Crops')
print('=' * 60)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(max(10, num_classes * 0.7), max(8, num_classes * 0.5)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title('Confusion Matrix - Odisha Crops', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 11. Upload Model to Vertex AI

This registers your trained model in Google Cloud Vertex AI Model Registry.

In [ ]:
# ============================================================
# 11. (OPTIONAL) Upload model to Vertex AI Model Registry
# ============================================================
# Needs real GCP credentials. If anything fails here it is caught and
# printed - the TF.js export below still runs. NOTE: the Keras model stays
# in the variable `model`; the registry entry goes into `vertex_model`
# (the old code overwrote `model` and broke the export cell).

vertex_model = None
try:
    model.save('/content/krishisetu_model.keras')
    print('Model saved locally: /content/krishisetu_model.keras')

    MODEL_BUCKET = f'{PROJECT_ID}-krishisetu'
    MODEL_PATH = 'models/krishisetu_odisha'

    from google.cloud import storage
    storage_client = storage.Client()
    try:
        bucket = storage_client.create_bucket(MODEL_BUCKET, location='us-central1')
        print(f'Created bucket: {MODEL_BUCKET}')
    except Exception:
        bucket = storage_client.get_bucket(MODEL_BUCKET)
        print(f'Using existing bucket: {MODEL_BUCKET}')

    blob = bucket.blob(f'{MODEL_PATH}/model.keras')
    blob.upload_from_filename('/content/krishisetu_model.keras')
    print(f'Model uploaded to gs://{MODEL_BUCKET}/{MODEL_PATH}/model.keras')

    vertex_model = aiplatform.Model.upload(
        display_name='krishisetu-odisha-crop-disease',
        artifact_uri=f'gs://{MODEL_BUCKET}/{MODEL_PATH}',
        serving_container_image_uri='us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-12:latest',
        description='KrishiSetu Odisha Crop Disease Classifier - MobileNetV2 Transfer Learning',
        labels={
            'project': 'krishisetu',
            'state': 'odisha',
            'hackathon': 'google-ai-2026'
        }
    )
    print(f'\nModel registered in Vertex AI: {vertex_model.resource_name}')
except Exception as e:
    print('\nVertex AI upload skipped (optional step). Reason:', e)


---
## 12. Quantize & Export to TensorFlow.js

In [ ]:
# ============================================================
# 12. Export to TensorFlow.js (Layers format for the offline PWA)
# ============================================================
# The app loads:  tf.loadLayersModel('/model/model.json')
# so we export Layers format: model.json + group1-shard*.bin + classes.json
# (the old cell fed a .tflite file into convert_tf_saved_model - that API
# call was wrong and produced nothing the app could load).

tfjs_dir = '/content/tfjs_model'
if os.path.exists(tfjs_dir):
    shutil.rmtree(tfjs_dir)
os.makedirs(tfjs_dir, exist_ok=True)

# Path A (preferred): Keras model -> TF.js Layers directly
try:
    tfjs.converters.save_keras_model(model, tfjs_dir)
    print('Exported via tfjs.converters.save_keras_model().')
except Exception as e:
    print('save_keras_model failed, falling back to SavedModel conversion:', e)
    sm_dir = '/content/keras_saved_model'
    if os.path.exists(sm_dir):
        shutil.rmtree(sm_dir)
    tf.saved_model.save(model, sm_dir)
    tfjs.converters.convert_tf_saved_model(sm_dir, tfjs_dir)
    print('Exported via convert_tf_saved_model().')

# classes.json MUST be a plain JSON array in the model's output order
# (the app reads classNames[argmax(probabilities)])
with open(os.path.join(tfjs_dir, 'classes.json'), 'w') as f:
    json.dump(class_names, f, indent=2)

metadata = {
    'model_name': 'KrishiSetu Odisha Crop Disease Classifier',
    'architecture': 'MobileNetV2 (alpha=1.0) + Transfer Learning',
    'input_size': [224, 224, 3],
    'input_range': [0.0, 1.0],           # app divides pixels by 255
    'num_classes': num_classes,
    'class_names': class_names,
    'quantization': 'float32',
    'google_ai_integration': {
        'vertex_ai_model': (vertex_model.resource_name if vertex_model
                            else 'skipped (optional step)'),
        'gemini_api': 'Cloud diagnosis fallback',
        'training_platform': 'Google Colab'
    },
    'training_datasets': {
        'plantvillage': 'https://www.kaggle.com/datasets/emmarex/plantdisease',
        'rice_disease': 'https://www.kaggle.com/datasets/anshulm257/rice-disease-dataset',
        'cotton_leaf': 'https://www.kaggle.com/datasets/seroshkarim/cotton-leaf-disease-dataset'
    },
    'state_focus': 'Odisha',
    'accuracy': float(max(history2.history['val_accuracy'])),
    'total_params': int(total_params)
}
with open(os.path.join(tfjs_dir, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

assert os.path.exists(os.path.join(tfjs_dir, 'model.json')), \
    'model.json missing - export failed!'

print('\nExport complete! Files in /content/tfjs_model:')
for fn in sorted(os.listdir(tfjs_dir)):
    print(f'  {fn:32s} {os.path.getsize(os.path.join(tfjs_dir, fn)) / 1e6:8.2f} MB')


In [ ]:
print('Exported files:')
print('-' * 50)
total_size = 0
for f in sorted(os.listdir(tfjs_dir)):
    fpath = os.path.join(tfjs_dir, f)
    fsize = os.path.getsize(fpath)
    total_size += fsize
    print(f'  {f:30s} {fsize/1e6:8.2f} MB')
print('-' * 50)
print(f'  TOTAL: {total_size/1e6:.2f} MB')

if total_size < 5e6:
    print(f'\nUnder 5MB! Perfect for mobile offline use.')
else:
    print(f'\nOver 5MB. Consider reducing classes or using alpha=0.75.')

---
## 13. Download the Model

In [ ]:
# ============================================================
# 13. Package + download the model zip
# ============================================================

shutil.make_archive('/content/tfjs_model', 'zip', tfjs_dir)
zip_size = os.path.getsize('/content/tfjs_model.zip') / 1e6
print(f'Created /content/tfjs_model.zip ({zip_size:.2f} MB)')

try:
    from google.colab import files
    files.download('/content/tfjs_model.zip')
except Exception:
    print('Auto-download unavailable - right-click tfjs_model.zip in the '
          'Files sidebar and choose Download.')

print()
print('DEPLOY INTO THE APP (on your computer, inside the KrishiSetu-AI repo):')
print('  mkdir -p public/model')
print('  unzip tfjs_model.zip')
print('  copy model.json, group1-shard*.bin, classes.json -> public/model/')
print()
print('  then run:  npm run dev')
print('  in the app: Settings -> Download Offline AI Model, then scan offline.')
print()
print('Google AI integration summary:')
print(f"  - Vertex AI model: {vertex_model.resource_name if vertex_model else 'skipped (optional)'}")
print('  - Training platform: Google Colab')
print('  - Cloud diagnosis: Gemini API')


---
## 14. Results Template

```
Model: KrishiSetu Odisha Crop Disease Classifier v1.0
Architecture: MobileNetV2 (alpha=1.0) + Transfer Learning
Quantization: INT8 post-training

Google AI Integration:
  - Vertex AI: Model trained and registered
  - Gemini API: Cloud-based crop diagnosis
  - Model Registry: [model.resource_name]

Results:
  Validation Accuracy: ____ %
  Model Size: ____ MB
  Classes: ____

Deployment:
  - Edge: TensorFlow.js in React PWA (offline)
  - Cloud: Vertex AI Endpoint (online backup)

Target: Low-end Android phones in Odisha
```